[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_7/L1.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 7 | BÀI 1


---

# **LỰA CHỌN MÔ HÌNH VÀ TINH CHỈNH SIÊU THAM SỐ**


|  |  |
|:---|:---|
|**Thời gian đọc** |  50 phút |
|**Kiến thức nền** | Học máy   |
|**Từ khóa** |Siêu tham số, tìm kiếm lưới, tìm kiếm ngẫu nhiên, kiểm định chéo  |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q mlxtend
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_7/"

*Ở các module trước, ta đã huấn luyện và đánh giá hiệu suất của các mô hình học máy. Mô hình học máy có một số tham số mà dữ liệu dạy cho nó; các tham số mô hình được tìm bằng cách khớp mô hình trên tập dữ liệu huấn luyện. Siêu tham số thì không thể học bằng quá trình huấn luyện thông thường và được cố định trước khi ta bắt đầu huấn luyện. Trong module này, ta sẽ học thêm về siêu tham số. Trong bài này, ta bắt đầu bằng việc bàn về cách chọn mô hình có hiệu suất tốt nhất và các chiến lược tinh chỉnh siêu tham số.*

## **1. Cải thiện hiệu suất mô hình**

Trước khi đưa (các) mô hình học máy vào vận hành, ta nên bảo đảm mô hình cho lời giải tốt nhất có thể, không chệch và khái quát hóa tốt trên dữ liệu mới chưa thấy.

Ta sẽ bàn về các chủ đề sau trong phần này.
- Quá khớp và thiếu khớp.
- Tập huấn luyện, tập kiểm định và tập kiểm tra.
- Kiểm định chéo.

Ta bắt đầu với việc chia dữ liệu thành các tập huấn luyện, kiểm tra và kiểm định.

### **1.1 Tập huấn luyện, kiểm định và kiểm tra**

Nên chia dữ liệu thành các tập huấn luyện, kiểm tra và kiểm định, nhất là khi ta muốn tinh chỉnh siêu tham số. Tập kiểm định giúp ta có đánh giá không chệch về hiệu suất mô hình và nhờ đó có thể tinh chỉnh siêu tham số của mô hình trước khi chuyển mô hình sang tập dữ liệu kiểm tra.

Với một tập dữ liệu, ta có thể có các tập huấn luyện, kiểm định và kiểm tra bằng một trong các cách sau:
 - Chia ngẫu nhiên
 - Chia phân tầng

Chia ngẫu nhiên là chia dữ liệu gốc theo các tỷ lệ định trước biểu diễn các tập huấn luyện, kiểm định và kiểm tra, bằng cách xáo trộn dữ liệu trước để vượt qua vấn đề tồn tại một mẫu hình dựa trên chỉ số của dữ liệu.

Chia phân tầng chia các tập huấn luyện, kiểm định và kiểm tra theo tỷ lệ các mẫu trong biến mục tiêu. Chia phân tầng thường dùng khi dữ liệu mất cân bằng.

Ta xem một ví dụ về điều này với tập dữ liệu nhà ở California, có biến mục tiêu là trung vị giá nhà.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
X, y = housing.data, housing.target * 10

In [ ]:
X.head()

In [ ]:
y.head()

Ta sẽ dùng bộ hồi quy cây quyết định để dự báo trung vị giá nhà, với mối quan tâm là khả năng khái quát hóa của mô hình. Vì vậy ta bắt đầu bằng việc huấn luyện mô hình trên toàn bộ tập dữ liệu.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

regressor = DecisionTreeRegressor(random_state=42)
regressor.fit(X, y)

Ta cũng đánh giá mô hình trên tập dữ liệu để xem nó hoạt động thế nào.

In [ ]:
from sklearn.metrics import mean_absolute_error

target_predicted = regressor.predict(X)
score = mean_absolute_error(y, target_predicted)
print(f"On average, our regressor makes an error of {score:.3f} ")

Sai số tuyệt đối trung bình thu được thực ra rất ấn tượng: nó cho biết mô hình của ta thực sự dự báo chính xác trung vị giá của từng căn nhà. Vậy câu hỏi là ta có nên chọn mô hình này không? Câu trả lời đơn giản là đừng vội! Hãy nhớ ta đã huấn luyện và đánh giá mô hình trên cùng một tập dữ liệu và bộ hồi quy cây quyết định có thể đã học thuộc tập dữ liệu, do đó cho ta quan niệm sai về khả năng khái quát hóa trên dữ liệu chưa thấy.

Khi huấn luyện mô hình, ta quan tâm đến việc nó cực tiểu hóa sai số trên dữ liệu chưa thấy, và đây là ý tưởng đằng sau việc ta chia dữ liệu thành tập huấn luyện và tập kiểm tra.

Vì vậy, ta có thể bắt đầu bằng việc chia tập dữ liệu như bên dưới. Một lần nữa, ta huấn luyện mô hình trên tập huấn luyện, đánh giá hiệu suất trên cùng dữ liệu đó và ghi lại sai số.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

Ta lại huấn luyện mô hình trên dữ liệu huấn luyện.

In [ ]:
regressor.fit(X_train, y_train)

Ta dự báo trên dữ liệu huấn luyện và xem sai số huấn luyện thu được.

In [ ]:
y_predicted = regressor.predict(X_train)
score = mean_absolute_error(y_train, y_predicted)
print(f"The training error of our model is {score:.3f}")

Kết quả cho thấy mô hình có thể dự báo chính xác giá trị nhà thực tế. Để kiểm tra độ mạnh của mô hình, ta dùng dữ liệu kiểm tra, vốn chưa được mô hình 'thấy', để dự báo trung vị giá nhà.

In [ ]:
y_predicted = regressor.predict(X_test)
score = mean_absolute_error(y_test, y_predicted)
print(f"The testing error of our model is {score:.3f} ")

Bây giờ ta thấy sai số trên tập dữ liệu kiểm tra lớn hơn so với tập huấn luyện và do đó có dấu hiệu mô hình đang *quá khớp*. Để tránh vấn đề quá khớp, ta có thể dùng kiểm định chéo, chia dữ liệu thành các phần nhỏ hơn theo cách lặp để bao phủ toàn bộ dữ liệu huấn luyện, và dữ liệu này lần lượt được chia thành tập huấn luyện và tập kiểm định.

### **1.2 Kiểm định chéo**

Trong kỹ thuật này, dữ liệu huấn luyện được chia thành tập huấn luyện và tập kiểm định nhiều lần, và mỗi lần chia gọi là một **phần** (Fold). Trong mục này, ta sẽ bàn về kiểm định chéo k lần.

Kiểm định chéo k lần chia dữ liệu thành $k$ đoạn, trong đó $k-1$ đoạn dùng để huấn luyện và đoạn còn lại dùng để kiểm định như trong sơ đồ bên dưới.

**Hình 1: Kiểm định chéo k lần**

<center><img src="https://ars.els-cdn.com/content/image/1-s2.0-S2666827021000906-gr5.jpg
" alt="Kiểm định chéo k lần" width="500"></center>

##### Nguồn: [Srivastava, Amiy, et al. "Ensemble Prediction of Mean Bubble Size in a Continuous Casting Mold Using Data Driven Modeling Techniques." *Machine Learning with Applications*, vol. 6, 2021.](https://www.sciencedirect.com/science/article/pii/S2666827021000906)


Dưới đây là các bước thực hiện kiểm định chéo:

1. Xáo trộn tập dữ liệu.
2. Giữ lại dữ liệu kiểm tra.
3. Thực hiện kiểm định chéo k lần trên tập dữ liệu huấn luyện.
4. Lấy điểm đánh giá tốt nhất, là trung bình của tất cả các phần.
5. Thực hiện đánh giá mô hình trên dữ liệu kiểm tra.

Kiểm định chéo cho phép ta ước lượng độ vững của mô hình bằng cách chia tập dữ liệu lặp lại một số lần cho trước.

Các ô tiếp theo minh họa cách áp dụng kiểm định chéo trên tập dữ liệu.

In [ ]:
from sklearn.model_selection import ShuffleSplit, cross_validate

cv = ShuffleSplit(n_splits=40, test_size=0.3, random_state=0)
cv_results = cross_validate(regressor, X, y, cv=cv, scoring="neg_mean_absolute_error")

Ta chuyển từ điển Python `cv_results` thành một dataframe để trực quan hóa dễ hơn.

In [ ]:
import pandas as pd

cv_results = pd.DataFrame(cv_results)
cv_results.head()

Ta đổi dấu `test_score` để có sai số kiểm tra thực tế, vì tham số `scoring="neg_mean_absolute_error"` cho giá trị âm của sai số tuyệt đối trung bình.

In [ ]:
cv_results["test_error"] = -cv_results["test_score"]

In [ ]:
cv_results.head(10)

Biểu đồ bên dưới cho thấy các sai số kiểm tra phân bố như thế nào.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(cv_results["test_error"], density=True)
# cv_results["test_error"].plot.hist(bins=10, edgecolor="black")
plt.xlabel("Mean absolute error")
plt.ylabel("Frequency Percentage %")
plt.suptitle(
    "Fig. 2: Test Errors Distribution.", fontweight="bold", horizontalalignment="right"
)
plt.show()

In [ ]:
print(
    f"The mean cross-validated testing error is: "
    f"{cv_results['test_error'].mean():.3f}"
)

Sai số đã giảm so với sai số ban đầu khi không dùng kiểm định chéo.

In [ ]:
print(
    f"The standard deviation of the testing error is: "
    f"{cv_results['test_error'].std():.3f}"
)

Kiểm định chéo làm giảm sai số của mô hình, dù ở mức nhỏ hơn so với trường hợp ta không dùng kiểm định chéo.

Một vấn đề thường xảy ra khi ta tìm cách cải thiện hiệu suất mô hình: ta có thể thấy mình giải quyết một vấn đề (thiếu khớp) trong khi tạo ra vấn đề khác (quá khớp). Ở mục tiếp theo, ta bàn về khái niệm quá khớp và thiếu khớp.

### **1.3 Quá khớp và thiếu khớp**

Khi mô hình học thuộc dữ liệu huấn luyện, ta nói đã xảy ra quá khớp. Trong trường hợp đó, ta có điểm đánh giá cao trên tập dữ liệu huấn luyện nhưng điểm không ấn tượng trên tập dữ liệu kiểm tra; do đó mô hình không khái quát hóa tốt trên dữ liệu chưa thấy.

Các mô hình quá khớp phản ứng với nhiễu ngẫu nhiên trong tập huấn luyện và do đó trở nên nhạy với những dao động nhỏ trong dữ liệu. Quá khớp là sự kết hợp của độ chệch thấp và phương sai cao.

Thiếu khớp, mặt khác, đặc trưng bởi điểm hiệu suất thấp ở cả tập dữ liệu huấn luyện và tập dữ liệu kiểm định. Thiếu khớp do các thuật toán đơn giản không học được các mẫu hình phức tạp có trong dữ liệu. Thiếu khớp cũng được gọi là độ chệch cao và thường có phương sai thấp.

Mô hình khớp đúng thì không thiếu khớp cũng không quá khớp, và hiệu suất mô hình trên tập huấn luyện tương đương tập kiểm tra. Các thuật toán học máy tuyến tính như hồi quy logistic có xu hướng thiếu khớp trên tập dữ liệu, trong khi các mô hình phi tuyến như hồi quy đa thức có xu hướng quá khớp.

Trong trường hợp mô hình thiếu khớp, để cải thiện hiệu suất, ta có thể
1. Dùng các thuật toán phi tuyến.
2. Tăng độ phức tạp của mô hình bằng cách điều chỉnh các tham số.
3. Dùng các thuật toán phi tham số.

Để hạn chế quá khớp của mô hình, ta
1. Tăng kích thước dữ liệu dùng để huấn luyện nhằm cải thiện khả năng khái quát hóa của mô hình.
2. Dùng các kỹ thuật điều chuẩn hóa như chuẩn L1 và L2.
3. Tinh chỉnh các siêu tham số của mô hình.
4. Giảm số đặc trưng trong tập dữ liệu.
5. Giảm độ phức tạp của mô hình.
6. Áp dụng các kỹ thuật kiểm định chéo.

**Hình 3: Ví dụ về quá khớp và thiếu khớp**

<center><img src="https://www.researchgate.net/publication/339680577/figure/fig2/AS:865364518924290@1583330387982/llustration-of-the-underfitting-overfitting-issue-on-a-simple-regression-case-Data.png" alt="Quá khớp và thiếu khớp" width="500"></center>

##### Nguồn: [Badillo, Solveig, et al. "An Introduction to Machine Learning." *Clinical Pharmacology & Therapeutics*, vol. 107, no. 4, 2020, pp. 871-885.](https://ascpt.onlinelibrary.wiley.com/doi/full/10.1002/cpt.1796)

Trong các ô tiếp theo, ta đánh giá mô hình và xác định nó khớp tốt, thiếu khớp hay quá khớp trên tập dữ liệu.

In [ ]:
import pandas as pd
from sklearn.model_selection import ShuffleSplit, cross_validate

cv = ShuffleSplit(n_splits=30, test_size=0.2)
cv_results = cross_validate(
    regressor,
    X,
    y,
    cv=cv,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
    n_jobs=2,
)
cv_results = pd.DataFrame(cv_results)

Như đã bàn ở mục trước, ta đổi dấu các điểm để có sai số huấn luyện và kiểm tra thực tế.

In [ ]:
scores = pd.DataFrame()
scores[["train error", "test error"]] = -cv_results[["train_score", "test_score"]]

Ô tiếp theo vẽ phân phối của các sai số.

In [ ]:
import matplotlib.pyplot as plt

scores.plot.hist(bins=50, edgecolor="black")
plt.xlabel("Mean absolute error")
plt.suptitle(
    "Fig. 4: Train and Test Errors Distribution via Cross-validation.",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Ta có thể quan sát thấy sai số huấn luyện bằng không, hàm ý mô hình của ta không thiếu khớp. Sai số kiểm tra lớn hơn, và do đó mô hình của ta có vẻ đã học thuộc nhiễu trong tập dữ liệu. Vì vậy khả năng khái quát hóa của nó bị ảnh hưởng.

Một trong những thách thức của tinh chỉnh siêu tham số là ta có thể dễ dàng chuyển từ trường hợp mô hình thiếu khớp sang trường hợp mô hình quá khớp trên tập dữ liệu. Để vượt qua những thách thức này, ta cần xét **đường cong kiểm định** (validation curve).

### **1.4 Đường cong kiểm định**

Đường cong kiểm định thay đổi giá trị của (các) siêu tham số và cho trực giác về cách mô hình hành xử với các giá trị siêu tham số khác nhau.

In [ ]:
from sklearn.model_selection import validation_curve

max_depth = [1, 3, 5, 10, 15, 20, 25, 30]
train_scores, test_scores = validation_curve(
    regressor,
    X,
    y,
    param_name="max_depth",
    param_range=max_depth,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
)
train_errors, test_errors = -train_scores, -test_scores

Trong ví dụ, ta giới hạn ở một trong các siêu tham số là `max_depth`.

In [ ]:
plt.plot(max_depth, train_errors.mean(axis=1), label="Training error")
plt.plot(max_depth, test_errors.mean(axis=1), label="Testing error")
plt.legend()

plt.xlabel("Maximum depth of decision tree")
plt.ylabel("Mean absolute error")
plt.suptitle(
    "Fig. 5: Validation Curve for Decision Tree",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Từ biểu đồ trên, ta thấy với các giá trị độ sâu tối đa thấp, mô hình có vẻ thiếu khớp, trong khi độ sâu tối đa của các cây tăng lên thì mô hình quá khớp trên tập dữ liệu. Vì vậy, với giá trị 10, ta có thể nói mô hình sẽ khớp tốt và do đó khái quát hóa tốt hơn trên tập dữ liệu chưa thấy. Biểu đồ bên dưới lặp lại biểu đồ trên nhưng cho các sai số huấn luyện.

In [ ]:
plt.errorbar(
    max_depth,
    train_errors.mean(axis=1),
    yerr=train_errors.std(axis=1),
    label="Training error",
)
plt.errorbar(
    max_depth,
    test_errors.mean(axis=1),
    yerr=test_errors.std(axis=1),
    label="Testing error",
)
plt.legend()

plt.xlabel("Maximum depth of decision tree")
plt.ylabel("Mean absolute error")
plt.suptitle(
    "Fig. 6: Validation Curve for Decision Tree using Train Errors",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Mục tiêu của ta khi chạy các mô hình học máy là có ước lượng không chệch về các sai số. Điều này khả thi khi ta hiểu khái niệm đánh đổi độ chệch và phương sai liên hệ thế nào với thiếu khớp và quá khớp.

### **1.5 Đánh đổi độ chệch và phương sai**

Ta có thể chia sai số dự báo của các thuật toán học máy thành hai phần:
- Phần có thể giảm
- Phần không thể giảm.

Phần không thể giảm do nhiễu ngẫu nhiên, do thiếu các đặc trưng quan trọng hoặc sai số đo lường, và không thể giảm dù mô hình rất tốt. Phần có thể giảm, mặt khác, gồm các sai số do:
- Độ chệch
- Phương sai

**Sai số do độ chệch** - Độ chệch là chênh lệch giữa các giá trị dự báo và các giá trị thực mà ta cố dự báo. Các sai số này xảy ra khi thuật toán quá đơn giản (ít phức tạp) để nắm bắt mối quan hệ hàm số giữa các đặc trưng. Mô hình có độ chệch cao có xu hướng đơn giản hóa mô hình quá mức do không học được nhiều từ dữ liệu huấn luyện. Điều này làm mô hình mắc một số sai lầm mang tính hệ thống, và do đó các dự báo sẽ bị chệch. Điều này cũng gọi là thiếu khớp.

**Sai số do phương sai** - Ta định nghĩa phương sai là thước đo độ phân tán trong tập dữ liệu. Các sai số này xảy ra khi thuật toán phức tạp đến mức trích xuất các mẫu hình từ nhiễu trong tập dữ liệu. Mô hình có phương sai cao hoạt động rất tốt trên dữ liệu huấn luyện nhưng không khái quát hóa tốt trên dữ liệu chưa thấy. Đây là cái ta gọi là quá khớp, như đã bàn ở trên.

Vậy ta thấy hai vấn đề ở đây:
1. Nếu mô hình quá đơn giản và có ít tham số thì có xu hướng ta có nguy cơ có mô hình với độ chệch cao và phương sai thấp.
2. Nếu mô hình quá phức tạp với nhiều tham số thì rất có khả năng mô hình sẽ quá khớp trên tập dữ liệu do phương sai cao và độ chệch thấp.

Đây là điều đưa ra sự đánh đổi độ chệch và phương sai, vì không thể có mô hình vừa phức tạp vừa đơn giản cùng lúc.

Xây dựng các mô hình chính xác đòi hỏi hiểu các sai số này để tránh mô hình thiếu khớp hoặc quá khớp.

Trong vài ô tiếp theo, ta sẽ minh họa khái niệm đánh đổi độ chệch và phương sai.

In [ ]:
from mlxtend.evaluate import bias_variance_decomp

Ta dùng dữ liệu như trên và cùng thuật toán để minh họa khái niệm đánh đổi độ chệch và phương sai.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
X, y = housing.data.values, housing.target.values * 10
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=1
)

Trong ô tiếp theo, ta sẽ định nghĩa siêu tham số xác định mức tăng độ phức tạp của mô hình.

In [ ]:
max_levels = list(range(1, 50))
levels = []
squared_bias_plus_variance = []
for level in max_levels:
    model = DecisionTreeRegressor(max_depth=level)
    model.fit(X_train, y_train)
    mse, bias, var = bias_variance_decomp(
        model,
        X_train,
        y_train,
        X_test,
        y_test,
        loss="mse",
        num_rounds=200,
        random_seed=1,
    )
    score = model.score(X_test, y_test)
    squared_bias_plus_variance.append(bias**2 + var)
    levels.append(level)

Trong ô tiếp theo, ta vẽ biểu đồ đánh đổi độ chệch và phương sai, cho thấy việc tăng độ phức tạp của mô hình ảnh hưởng đến độ chệch và phương sai thế nào.

In [ ]:
import plotly.graph_objects as go

scatter = go.Scatter(x=levels, y=squared_bias_plus_variance)
layout = go.Layout(
    title="Fig. 7. Bias variance tradeoff",
    xaxis=dict(title="levels"),
    yaxis=dict(title="bias^2+variance"),
)
go.Figure(data=[scatter], layout=layout)

Có thể thấy khi độ phức tạp của mô hình tăng, sai số giảm.

Bây giờ ta đã hiểu rõ cách cải thiện hiệu suất mô hình, ta có thể tiếp tục học cách điều chỉnh các siêu tham số của mô hình.

## **2. Tinh chỉnh siêu tham số**

Tinh chỉnh siêu tham số gồm việc tìm kiếm trên một tập giá trị cho trước và tìm giá trị tối ưu của một siêu tham số trong mô hình học máy. Ta kỳ vọng sau khi tinh chỉnh siêu tham số, kết quả huấn luyện mô hình sẽ cho điểm tốt hơn mà không quá khớp trên tập dữ liệu.

Bây giờ ta nêu bật khác biệt giữa tham số và siêu tham số.

Trong khi tham số được học từ dữ liệu trong quá trình huấn luyện mô hình, siêu tham số do người dùng đặt trước khi ta bắt đầu huấn luyện mô hình.

Ví dụ về tham số là các hệ số ta có được từ việc huấn luyện mô hình hồi quy tuyến tính, còn số epoch và kích thước lô là ví dụ về siêu tham số. Cần lưu ý một số mô hình như hồi quy tuyến tính chỉ có tham số, trong khi các mô hình khác như K láng giềng gần nhất (KNN) chỉ chứa siêu tham số.

Trước khi bắt đầu tinh chỉnh siêu tham số, trước hết ta định nghĩa **không gian siêu tham số**, là tập mọi giá trị siêu tham số có thể. Siêu tham số có thể là giá trị rời rạc hoặc liên tục. Khi định nghĩa không gian siêu tham số, ta cũng định nghĩa phân phối nền tảng của chúng.

Ở các mục tiếp theo của bài này, ta định bàn về các loại tinh chỉnh siêu tham số.

### **2.1 Tìm kiếm thủ công**

Phương pháp tìm kiếm thủ công gồm việc định nghĩa không gian tìm kiếm và duyệt qua mọi siêu tham số có thể để có một tổ hợp cho kết quả tốt nhất. Ta theo các bước sau khi thực hiện tìm kiếm thủ công:

1. Chia dữ liệu huấn luyện thành các phần huấn luyện và kiểm tra.
2. Định nghĩa các giá trị siêu tham số ban đầu.
3. Thực hiện kiểm định chéo k lần trên dữ liệu huấn luyện.
4. Lấy điểm kiểm định.
5. Chọn một bộ giá trị siêu tham số khác.
6. Lặp lại các bước 3-5 đến khi có hiệu suất thỏa đáng.
7. Huấn luyện mô hình trên toàn bộ tập huấn luyện với giá trị siêu tham số đã chọn.
8. Đánh giá hiệu suất này trên tập kiểm tra.

Trong ví dụ bên dưới, ta cho thấy cách lấy và đặt (các) giá trị siêu tham số của mô hình. Ta sẽ xét [dữ liệu hiến máu](https://archive.ics.uci.edu/ml/machine-learning-databases/blood-transfusion/transfusion.data).

In [ ]:
from sklearn import set_config

set_config(display="diagram")

import pandas as pd

Ta nạp dữ liệu và hiển thị vài dòng đầu.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/blood-transfusion/transfusion.data"
data_df = pd.read_csv(url)
data_df.head()

Bước tiếp theo là đổi tên các cột để xử lý dễ hơn.

In [ ]:
cols = [c.lower().split()[0] for c in data_df.columns]
cols[-1] = "class_name"
data_df.columns = cols

In [ ]:
data_df.head()

Sau đó ta định nghĩa nhóm mục tiêu, thiết lập loại bài toán học máy mà ta cần giải. Trong trường hợp của ta, ta có bài toán phân loại nhị phân và sẽ không đi sâu vì ta chỉ quan tâm đến việc học các kỹ thuật tinh chỉnh siêu tham số.

In [ ]:
target_name = "class_name"
feature_columns = ["recency", "frequency", "monetary"]

target = data_df[target_name]
data = data_df[feature_columns]

Ta biến đổi dữ liệu bằng cách co giãn lại bằng `StandardScaler`, loại bỏ trung bình của đặc trưng và co giãn về phương sai đơn vị.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline(
    steps=[
        ("preprocessor", StandardScaler()),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

Như đã bàn ở phần trước, ta đánh giá hiệu suất của mô hình bằng kiểm định chéo.

In [ ]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, data, target)
scores = cv_results["test_score"]
print(
    f"Accuracy score via cross-validation:\n"
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

Trong các phần minh họa tiếp theo của bài này, ta sẽ thử tinh chỉnh các siêu tham số `learning rate` và `max_leaf_nodes` của mô hình `HistGradientBoostingClassifier`. Trước hết ta hiển thị các giá trị mặc định của chúng bên dưới.

In [ ]:
print("learning rate default value", model.get_params()["classifier__learning_rate"])
print("max_leaf_nodes default value", model.get_params()["classifier__max_leaf_nodes"])

Khi không chỉ định giá trị của các siêu tham số, mô hình dùng các tham số mặc định, ví dụ giá trị mặc định của C là 1. Ta có thể tinh chỉnh thủ công siêu tham số, trong trường hợp này là đổi giá trị C như trong ô bên dưới.

In [ ]:
model.set_params(classifier__learning_rate=1e-3)
model.set_params(classifier__max_leaf_nodes=20)
cv_results = cross_validate(model, data, target)
scores = cv_results["test_score"]
print(
    f"Model accuracy score with cross-validation:\n"
    f"{scores.mean():.3f} ± {scores.std():.3f}"
)

Các siêu tham số được lưu trong `model.get_params` và ta có thể khám phá từng cái một. Để minh họa, ta sẽ xét tinh chỉnh siêu tham số `C` trong các ô bên dưới.

Ô bên dưới hiển thị giá trị của các siêu tham số ta đã đặt ở trên.

In [ ]:
print("learning rate default value", model.get_params()["classifier__learning_rate"])
print("max_leaf_nodes default value", model.get_params()["classifier__max_leaf_nodes"])

Trong ô tiếp theo, ta đặt thủ công một vài giá trị cho C và chọn giá trị cho hiệu suất tốt hơn.

In [ ]:
for learning_rate in [1e-3, 1e-2, 1e-1, 1, 10]:
    for max_leaf_nodes in [3, 5, 10, 15, 20, 30]:
        model.set_params(classifier__learning_rate=learning_rate)
        model.set_params(classifier__max_leaf_nodes=max_leaf_nodes)
        cv_results = cross_validate(model, data, target)
        scores = cv_results["test_score"]
        print(
            f"Model accuracy score with cross-validation:\n learning_rate={learning_rate} and max_leaf_nodes={max_leaf_nodes}:\n"
            f"{scores.mean():.3f} ± {scores.std():.3f}"
        )

Từ ô trên, ta thấy giá trị `C=0,01` cho độ chính xác tốt hơn.

### **2.2 Tìm kiếm lưới**

Với một không gian siêu tham số, tìm kiếm lưới xem xét mọi tổ hợp có thể cải thiện hiệu suất của mô hình. Trong phương pháp này, không cần biết phân phối nền tảng của các siêu tham số.

Vậy các bước trong tìm kiếm lưới như sau.
1. Chia dữ liệu thành các tập huấn luyện và kiểm tra.
2. Định nghĩa không gian siêu tham số.
3. Dựng vòng lặp lồng nhau theo số siêu tham số.
4. Thực hiện kiểm định chéo trên tập huấn luyện và lưu điểm thu được ở mỗi vòng lặp.
5. Huấn luyện mô hình trên tập huấn luyện và đánh giá trên tập kiểm tra.

Kết quả tìm kiếm lưới sẽ bằng kết quả tìm kiếm thủ công vì mô hình được đánh giá tại mỗi cặp siêu tham số trong lưới của ta.

Như ở phần tinh chỉnh thủ công ở trên, ta minh họa cách áp dụng tìm kiếm lưới trong tinh chỉnh siêu tham số.

Ta bắt đầu bằng việc chia tập dữ liệu thành các tập huấn luyện và kiểm tra.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data, target, random_state=42)

Ta dùng mô hình bộ phân loại gradient dạng histogram, là bộ phân loại dựa trên cây.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

model = Pipeline(
    [
        ("preprocessor", StandardScaler()),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)
model

Tìm kiếm lưới tốn kém vì nó tìm một tổ hợp của mọi siêu tham số trong không gian siêu tham số. Trong ví dụ này, ta chỉ chia sẻ số nút giới hạn, và ta chọn chỉ tinh chỉnh hai siêu tham số.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__learning_rate": (0.001, 0.01, 0.1, 1, 10),
    "classifier__max_leaf_nodes": (3, 5, 10, 15, 20, 30),
}
model_grid_search = GridSearchCV(model, param_grid=param_grid, n_jobs=2, cv=2)
model_grid_search.fit(X_train, y_train)

Sau đó ta đánh giá độ chính xác của mô hình bằng tập kiểm tra.

In [ ]:
accuracy = model_grid_search.score(X_test, y_test)
print(f"The test accuracy score of the grid-searched pipeline is: " f"{accuracy:.3f}")

Tìm kiếm lưới lấy không gian siêu tham số chứa mọi giá trị siêu tham số và tìm mọi tổ hợp có thể của các siêu tham số. Như trong trường hợp của ta có bốn giá trị learning_rate và max_leaf_nodes, sẽ có $4\times 4 = 16$ tổ hợp siêu tham số, và ta chọn cặp cho kết quả tốt nhất. Điều này làm quá trình tốn kém vì với mỗi giá trị ta thêm vào, quá trình càng trở nên không bền vững về mặt tính toán. Ta thực sự có thể thấy tìm kiếm lưới tương tự tìm kiếm thủ công mà ta thấy ở mục trước.

Từ các siêu tham số đã chọn, bây giờ ta có thể dự báo năm mục tiêu đầu.

In [ ]:
model_grid_search.predict(X_test.iloc[0:5])

Bây giờ ta hiển thị các giá trị gốc trong ô bên dưới.

In [ ]:
y_test.iloc[0:5].values

Bằng cách dùng thuộc tính `best_params_`, ta có thể thấy các siêu tham số được chọn cho độ chính xác cao nhất.

In [ ]:
print(f"The best set of parameters is: " f"{model_grid_search.best_params_}")

Ta xem xét mọi giá trị lưu trong cv_results_ bằng một dataframe như bên dưới.

In [ ]:
cv_results = pd.DataFrame(model_grid_search.cv_results_).sort_values(
    "mean_test_score", ascending=False
)
cv_results.head()

Ta rút gọn các tiêu đề để dataframe dễ đọc hơn.

In [ ]:
# get the parameter names
column_results = [f"param_{name}" for name in param_grid.keys()]
column_results += ["mean_test_score", "std_test_score", "rank_test_score"]
cv_results = cv_results[column_results]

In [ ]:
def shorten_param(param_name):
    if "__" in param_name:
        return param_name.rsplit("__", 1)[1]
    return param_name


cv_results = cv_results.rename(shorten_param, axis=1)
cv_results.head()

Vì ta chỉ quan tâm đến hai siêu tham số của mình, bảng bên dưới hiển thị độ chính xác của mô hình ở mỗi cặp.

In [ ]:
pivoted_cv_results = cv_results.pivot_table(
    values="mean_test_score", index=["learning_rate"], columns=["max_leaf_nodes"]
)

pivoted_cv_results

Ta có thể trực quan hóa bảng trên bằng biểu đồ nhiệt như ở ô tiếp theo.

In [ ]:
import seaborn as sns

ax = sns.heatmap(pivoted_cv_results, annot=True, cmap="YlGnBu", vmin=0.0, vmax=0.9)
ax.invert_yaxis()
plt.suptitle(
    "Fig. 8: Heatmap of the Model Results",
    fontweight="bold",
    horizontalalignment="right",
)
plt.show()

Thách thức lớn nhất ta gặp với hai phương pháp trên là chúng tốn rất nhiều tính toán, nhất là khi không gian siêu tham số lớn. Ở mục tiếp theo, ta sẽ bàn về phương pháp tìm kiếm ngẫu nhiên, cố gắng giải quyết vấn đề này.

### **2.3 Tìm kiếm ngẫu nhiên**

Phương pháp này chọn siêu tham số ngẫu nhiên cho mỗi vòng lặp. Nó vượt trội tìm kiếm lưới khi dữ liệu nhỏ hoặc ta không biết phạm vi của không gian siêu tham số. Nó cũng hiệu quả hơn nhiều so với tìm kiếm lưới. Trong tìm kiếm ngẫu nhiên, phân phối siêu tham số phải được chỉ định cùng với không gian siêu tham số, và không gian này rộng hơn so với trường hợp tìm kiếm lưới.

Các bước trong tìm kiếm ngẫu nhiên là:
1. Chia dữ liệu thành các tập huấn luyện và kiểm tra.
2. Định nghĩa số lần thử và đặt hạt giống ngẫu nhiên.
3. Định nghĩa không gian siêu tham số cùng phân phối của nó.
4. Định nghĩa một bộ lặp gồm các tổ hợp siêu tham số ngẫu nhiên.
5. Lặp qua bộ lặp.
6. Sau khi có tổ hợp siêu tham số tốt nhất có thể, ta huấn luyện mô hình trên toàn bộ tập huấn luyện và đánh giá hiệu suất mô hình.


Trong các ô sau, ta minh họa phương pháp tìm kiếm ngẫu nhiên để tinh chỉnh siêu tham số trên dữ liệu hiến máu của ta.

In [ ]:
model = Pipeline(
    [
        ("preprocessor", StandardScaler()),
        ("classifier", HistGradientBoostingClassifier(random_state=42)),
    ]
)

Ta có thể xem các giá trị siêu tham số mặc định trong ô bên dưới.

In [ ]:
from pprint import pprint

print("Default parameters:\n")
pprint(model.get_params())

Ô tiếp theo định nghĩa không gian tìm kiếm siêu tham số.

In [ ]:
from scipy.stats import loguniform


class loguniform_int:
    """Integer valued version of the log-uniform distribution"""

    def __init__(self, a, b):
        self._distribution = loguniform(a, b)

    def rvs(self, *args, **kwargs):
        """Random variable sample"""
        return self._distribution.rvs(*args, **kwargs).astype(int)


from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__learning_rate": loguniform(1e-6, 1e3),
    "classifier__max_leaf_nodes": loguniform_int(3, 100),
}

Bây giờ ta có thể huấn luyện mô hình trên dữ liệu huấn luyện bằng kiểm định chéo thực hiện 100 vòng lặp với 5 phần.

In [ ]:
model_random_search = RandomizedSearchCV(
    model,
    param_distributions=param_distributions,
    n_iter=10,
    cv=5,
    verbose=1,
    n_jobs=-1,
    return_train_score=True,
)
model_random_search.fit(X_train, y_train)

Phương pháp tìm kiếm ngẫu nhiên để tinh chỉnh siêu tham số chọn các tổ hợp siêu tham số ngẫu nhiên, trái với tìm kiếm lưới, nơi mỗi giá trị trong không gian siêu tham số được dùng tường minh. Trong tìm kiếm ngẫu nhiên, việc chọn một giá trị siêu tham số độc lập với các giá trị còn lại. Trái với tìm kiếm lưới, nơi không gian siêu tham số gồm các giá trị siêu tham số, tìm kiếm ngẫu nhiên cho phép dùng các phân phối lấy mẫu.

Ô tiếp theo hiển thị các tham số tốt nhất từ mô hình của ta.

In [ ]:
model_random_search.best_params_

Ta đánh giá hiệu suất mô hình trên tập dữ liệu kiểm tra; tuy nhiên ta không kỳ vọng kết quả tốt vì các siêu tham số được chọn tùy ý nhằm mục đích minh họa phương pháp tìm kiếm ngẫu nhiên.

In [ ]:
accuracy = model_random_search.score(X_test, y_test)

print(f"The test accuracy score of the best model is " f"{accuracy:.3f}")

Như quá trình tìm kiếm lưới ở trên, ta xem xét kết quả kiểm định chéo ở ô tiếp theo.

In [ ]:
cv_results_rs = pd.DataFrame(model_random_search.cv_results_).sort_values(
    "mean_test_score", ascending=False
)
cv_results_rs.head()

In [ ]:
def shorten_param(param_name):
    if "__" in param_name:
        return param_name.rsplit("__", 1)[1]
    return param_name


cv_results_rs = cv_results_rs.rename(shorten_param, axis=1)
cv_results_rs.head()

Bảng bên dưới hiển thị các cặp mà tinh chỉnh bằng tìm kiếm ngẫu nhiên đã chọn. Vì `n_iter=10`, tìm kiếm ngẫu nhiên sẽ chọn ngẫu nhiên 10 cặp.

In [ ]:
pivoted_cv_results_rs = cv_results_rs.pivot_table(
    values="mean_test_score", index=["learning_rate"], columns=["max_leaf_nodes"]
)

pivoted_cv_results_rs

Ta có thể quan sát thấy việc tinh chỉnh siêu tham số bằng tìm kiếm ngẫu nhiên không đánh giá mô hình ở mọi cặp mà chọn ngẫu nhiên tổ hợp tốt nhất có thể trong các cặp. Vì tìm kiếm ngẫu nhiên không chọn mọi cặp có thể, nó có xu hướng nhanh hơn tìm kiếm thủ công và tìm kiếm lưới, và độ chính xác gần bằng tìm kiếm lưới.

### **2.4 Tối ưu Bayes**

Tối ưu Bayes cải thiện không gian tìm kiếm dựa trên những gì nó học được từ các vòng lặp trước. Trong kiểu chọn siêu tham số này, trước hết ta xây dựng mô hình xác suất của hàm mục tiêu rồi chọn các siêu tham số tối ưu.

Có nhiều kỹ thuật tối ưu Bayes được liệt kê dưới đây.
- Quá trình Gauss (GP).
- Tối ưu hóa dựa trên mô hình tuần tự
- Tree Parzen Estimator (TPE)

Tối ưu Bayes ghi nhớ hiệu suất quá khứ của mô hình, dùng để tìm xác suất của một điểm số cho trước một tham số, tức là $P(\text{score}|\text{hyperparameters})$.

Mô hình có thể được tối ưu dễ dàng và dùng để tìm (các) siêu tham số tiếp theo.

Các bước của tối ưu Bayes là:
1. Tìm mô hình xác suất của một điểm số cho trước một siêu tham số.
2. Tìm các siêu tham số tối ưu của mô hình ở (1).
3. Đưa các siêu tham số vào hàm mục tiêu.
4. Cập nhật mô hình xác suất với kết quả mới.
5. Lặp lại các bước 1 đến 4 đến khi có kết quả thỏa đáng.

Ta sẽ bàn thêm về tối ưu Bayes ở bài 4.

## **3. Kết luận**

Trong bài này, ta đã thấy các kỹ thuật khác nhau để cải thiện hiệu suất mô hình và tác động của siêu tham số lên hiệu suất mô hình. Ta cũng đã bàn về cách tự động hóa việc tìm siêu tham số bằng tìm kiếm lưới và tìm kiếm ngẫu nhiên. Ở bài tiếp theo, ta sẽ áp dụng các phương pháp này vào một ví dụ thực tế.

**Tài liệu tham khảo**

1. Badillo, Solveig, et al. "An Introduction to Machine Learning." *Clinical Pharmacology & Therapeutics*, vol. 107, no. 4, 2020, pp. 871-885.
2. Srivastava, Amiy, et al. "Ensemble Prediction of Mean Bubble Size in a Continuous Casting Mold Using Data Driven Modeling Techniques." *Machine Learning with Applications*, vol. 6, 2021.
3. Yang, Li, and Abdallah Shami. "On Hyperparameter Optimization of Machine Learning Algorithms: Theory and Practice." *Neurocomputing*, vol. 415, 2020, pp. 295-316.